# Semaine 1 — Exploration des données de capteurs de stationnement (Melbourne 2019)

**Objectif :** comprendre à quoi ressemblent ces données et tracer l'occupation d'une place dans le temps.

On *prototype* ici le chargement (lecture par chunks depuis le zip + échantillonnage). Une fois stabilisé, ce code sera remonté dans `parking.ingestion` (semaine 4).

> ⚠️ Le fichier brut fait ~42,7 M lignes (717 Mo compressé). On ne le charge **jamais** en entier : lecture par chunks + échantillon à 2 %.

In [ ]:
import re
import zipfile

import matplotlib.pyplot as plt
import pandas as pd

from parking.config import DATA_RAW

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

ZIP_PATH = DATA_RAW / "on-street-parking-2019.zip"
ZIP_PATH, ZIP_PATH.exists()

## 1. Quelles colonnes ? (lecture de l'en-tête seulement)

On ouvre le zip et on lit *uniquement* la première ligne du CSV pour connaître le schéma, sans rien charger.

In [ ]:
def find_csv_member(zf: zipfile.ZipFile) -> str:
    """Nom du vrai CSV (on ignore les fichiers parasites macOS __MACOSX/._*)."""
    return next(
        n
        for n in zf.namelist()
        if n.lower().endswith(".csv") and not n.split("/")[-1].startswith("._")
    )


with zipfile.ZipFile(ZIP_PATH) as zf:
    member = find_csv_member(zf)
    with zf.open(member) as f:
        header = pd.read_csv(f, nrows=0)

print("Fichier dans le zip :", member)
print("Colonnes brutes :")
list(header.columns)

## 2. Chargement d'un échantillon (2 %) par chunks

On lit le CSV par blocs de 500 000 lignes (contrôle la RAM) et on garde 2 % de chaque bloc au hasard. Résultat : ~850 k lignes, suffisant pour explorer, instantané à manipuler.

On normalise aussi les noms de colonnes en `snake_case` et on parse les dates.

In [ ]:
def to_snake(name: str) -> str:
    s = re.sub(r"(?<!^)(?=[A-Z])", "_", name.strip())
    return re.sub(r"[\s\-]+", "_", s).lower()


FRAC = 0.02
CHUNKSIZE = 500_000
SEED = 42
# Format des dates Melbourne : ex. "04/16/2019 02:14:47 PM" (mois en premier, AM/PM)
DATE_FMT = "%m/%d/%Y %I:%M:%S %p"

parts = []
n_total = 0
with zipfile.ZipFile(ZIP_PATH) as zf, zf.open(find_csv_member(zf)) as f:
    for i, chunk in enumerate(pd.read_csv(f, chunksize=CHUNKSIZE, low_memory=False)):
        n_total += len(chunk)
        parts.append(chunk.sample(frac=FRAC, random_state=SEED + i))

df = pd.concat(parts, ignore_index=True)
df.columns = [to_snake(c) for c in df.columns]

# Parse des colonnes temporelles (arrivée / départ) avec le format explicite
dt_cols = [c for c in df.columns if re.search(r"arrival|departure", c, re.I)]
for c in dt_cols:
    df[c] = pd.to_datetime(df[c], format=DATE_FMT, errors="coerce")

print(f"Échantillon : {len(df):,} lignes sur ~{n_total:,} ({FRAC:.0%})")
print("Colonnes temporelles parsées :", dt_cols)
df.head()

## 3. À quoi ressemblent ces données ?

In [ ]:
print("Dimensions :", df.shape)
df.info()

In [ ]:
df.describe(include="all").T

## 4. Qualité des données

Le dataset est connu pour ses erreurs de capteurs : valeurs manquantes, doublons, et surtout **départs antérieurs aux arrivées** (à nettoyer en semaine 2).

In [ ]:
# Valeurs manquantes
na = df.isna().mean().sort_values(ascending=False)
print("Taux de valeurs manquantes par colonne :")
print((na[na > 0] * 100).round(2).astype(str) + " %" if (na > 0).any() else "Aucune")

# Doublons
print("\nLignes dupliquées :", df.duplicated().sum())

In [ ]:
# Détection des colonnes arrivée / départ
arr_col = next((c for c in df.columns if "arrival" in c), None)
dep_col = next((c for c in df.columns if "departure" in c), None)
print("Colonne arrivée :", arr_col, "| colonne départ :", dep_col)

if arr_col and dep_col:
    bad = df[dep_col] < df[arr_col]
    print(f"Départs avant arrivées (erreur capteur) : {bad.sum():,} ({bad.mean():.2%})")
    # Durée de stationnement (minutes) sur les lignes valides
    duree = (df.loc[~bad, dep_col] - df.loc[~bad, arr_col]).dt.total_seconds() / 60
    print(duree.describe())

## 5. Occupation d'une place dans le temps (le « Done quand » ✅)

On choisit **une place** (capteur) avec beaucoup d'événements, et on trace son état occupé (1) / libre (0) sur une journée, à partir des intervalles arrivée→départ.

In [ ]:
# Identifiant d'une place : on préfère street_marker (repère physique de la place),
# sinon bay_id, sinon device_id (le capteur).
id_col = next((c for c in ["street_marker", "bay_id", "device_id"] if c in df.columns), None)
print("Identifiant de place :", id_col)

valid = df[(df[dep_col] >= df[arr_col])].dropna(subset=[arr_col, dep_col, id_col])
top_place = valid[id_col].value_counts().index[0]

# Un jour donné pour cette place
one = valid[valid[id_col] == top_place].copy()
jour = one[arr_col].dt.date.value_counts().index[0]
one_day = one[one[arr_col].dt.date == jour].sort_values(arr_col)
print(f"Place {top_place} — journée du {jour} — {len(one_day)} stationnements")
one_day[[arr_col, dep_col]].head()

In [ ]:
# Courbe en escalier : 1 = occupé, 0 = libre
fig, ax = plt.subplots(figsize=(12, 3))
for _, row in one_day.iterrows():
    ax.fill_betweenx([0, 1], row[arr_col], row[dep_col], color="tab:red", alpha=0.6)
ax.set_ylim(0, 1)
ax.set_yticks([0, 1])
ax.set_yticklabels(["libre", "occupé"])
ax.set_title(f"Occupation de la place {top_place} le {jour}")
ax.set_xlabel("Heure")
plt.tight_layout()
plt.show()

## 6. Premiers patterns : occupation moyenne par heure de la journée

Aperçu rapide de la dynamique globale (les vrais agrégats par rue/tranche viendront en semaine 2).

In [ ]:
# Nombre d'arrivées par heure (proxy de la demande)
par_heure = valid[arr_col].dt.hour.value_counts().sort_index()
ax = par_heure.plot(kind="bar", figsize=(12, 3), color="tab:blue")
ax.set_title("Nombre d'arrivées par heure de la journée (échantillon 2019)")
ax.set_xlabel("Heure")
ax.set_ylabel("Arrivées")
plt.tight_layout()
plt.show()

## 7. Conclusions de l'exploration

**Taille & couverture**
- Échantillon : **853 455 lignes** (2 % de **42 672 743** lignes pour l'année 2019).
- Granularité : 1 ligne = 1 stationnement (arrivée → départ) sur une place équipée d'un capteur.
- Couverture : **123 rues**, **4 973 places** (`street_marker`), **6 326 capteurs** (`device_id`) — il y a plus de capteurs que de places car certains ont été remplacés dans l'année.
- Colonnes clés pour le ML : `street_marker` / `street_name` (la place / la rue), `arrival_time` & `departure_time` (les intervalles d'occupation), `duration_minutes`.

**Qualité des données**
- **Pas de doublons.**
- Manquants : `sign` / `sign_plate_id` à **35 %** (info panneau/restriction, pas bloquant), le reste < 0,1 %.
- ⚠️ Surprise vs plan : l'erreur « départ avant arrivée » est **quasi inexistante** ici (**3 lignes**). Les vrais problèmes de qualité sont ailleurs :
  - **446 durées ≤ 0 min** (capteurs qui se déclenchent/relâchent instantanément),
  - **durées aberrantes** : max ≈ **71 577 min (~49 jours)** → capteurs bloqués à nettoyer.
- Durée de stationnement : médiane **9 min**, moyenne **49 min** (distribution très asymétrique, tirée par les longues durées).

**Signal d'occupation**
- `vehicle_present = True` sur **52,9 %** des lignes ; `in_violation` sur **3,7 %**.
- Pattern horaire net : **pic à 18 h**, creux à 4 h, top 3 heures = **18 h, 7 h, 17 h** → cohérent avec les trajets domicile-travail. Bon signe : il y a un vrai signal temporel à apprendre.

**Ce qu'on garde pour le ML** : `street_marker`, `street_name`, `arrival_time`, `departure_time` (+ heure / jour de semaine dérivés).

**Prochaine étape (semaine 2)** : nettoyage (durées ≤ 0 et aberrantes), reconstruction de l'état occupé/libre à un instant t, agrégation en taux d'occupation **par rue et tranche de 30 min**, puis EDA des patterns par heure / jour / rue.